In [2]:
# =========================
# LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries loaded")

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("data/processed/clean_reviews.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower()

print("Data loaded:", df.shape)
print(df.head())

# =========================
# STANDARDIZE BANK NAME
# =========================
df["bank"] = df["bank"].astype(str).str.upper()

df["bank"] = df["bank"].replace({
    "COMMERCIAL BANK OF ETHIOPIA (CBE)": "CBE",
    "COMMERCIAL BANK OF ETHIOPIA": "CBE",
    "BOA": "BOA",
    "BANK OF ABYSSINIA": "BOA",
    "DASHEN BANK": "DASHEN"
})

# =========================
# SENTIMENT MODEL
# =========================
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

texts = df["review"].astype(str).tolist()
results = sentiment_model(texts)

df["sentiment_label"] = [r["label"].lower() for r in results]
df["sentiment_score"] = [r["score"] for r in results]

# =========================
# CLEAN TEXT
# =========================
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df["clean"] = df["review"].apply(clean)

# =========================
# THEME CLASSIFICATION
# =========================
def assign_theme(text):
    text = str(text).lower()

    if any(x in text for x in ["login", "otp", "password"]):
        return "Account Access Issues"

    if any(x in text for x in ["transfer", "transaction", "slow", "delay"]):
        return "Transaction Performance"

    if any(x in text for x in ["ui", "design", "interface"]):
        return "UI & Experience"

    if any(x in text for x in ["error", "bug", "crash"]):
        return "App Stability"

    if any(x in text for x in ["feature", "request", "add"]):
        return "Feature Requests"

    return "Other"

df["identified_theme"] = df["clean"].apply(assign_theme)

# =========================
# FINAL OUTPUT FOR TASK 3
# =========================
df["review_id"] = df.index.astype(str)
df["review_date"] = df["date"]

df["source"] = df["source"].str.lower()

final_df = df[[
    "review_id",
    "bank",
    "review",
    "rating",
    "review_date",
    "sentiment_label",
    "sentiment_score",
    "identified_theme",
    "source"
]]

# Save for Task 3
final_df.to_csv("data/processed/task2_final_output.csv", index=False)

# =========================
# ANALYSIS
# =========================
print("\nSentiment by Bank:")
print(df.groupby("bank")["sentiment_score"].mean())

print("\nTheme Distribution:")
print(df.groupby(["bank", "identified_theme"]).size())

print("\nTASK 2 COMPLETE ✔")
print("Saved: task2_final_output.csv")

Libraries loaded
Data loaded: (1500, 5)
                                              review  rating        date  \
0                                   Good application       2  2026-05-13   
1  Very Secure but very poor interface and limite...       1  2026-05-13   
2                                     very nice 100%       5  2026-05-13   
3  Nice, but I can't get some recently transactio...       5  2026-05-13   
4                                               nice       5  2026-05-13   

                                bank       source  
0  Commercial Bank of Ethiopia (CBE)  Google Play  
1  Commercial Bank of Ethiopia (CBE)  Google Play  
2  Commercial Bank of Ethiopia (CBE)  Google Play  
3  Commercial Bank of Ethiopia (CBE)  Google Play  
4                        Dashen Bank  Google Play  


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Sentiment by Bank:
bank
BANK OF ABYSSINIA (BOA)    0.966775
CBE                        0.976784
DASHEN                     0.976074
Name: sentiment_score, dtype: float64

Theme Distribution:
bank                     identified_theme       
BANK OF ABYSSINIA (BOA)  Account Access Issues       15
                         App Stability               15
                         Feature Requests             4
                         Other                      431
                         Transaction Performance     27
                         UI & Experience              8
CBE                      Account Access Issues        1
                         App Stability               11
                         Feature Requests             5
                         Other                      444
                         Transaction Performance     34
                         UI & Experience              5
DASHEN                   Account Access Issues        6
                         App St